# Практическая работа № 2 (пример выполнения)
## Иерархия игровых сущностей: ABC, протоколы, dataclass, миксины

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 2 — Углублённое ООП на Python  
**Проект:** «Тайна исчезнувшего дива» — мистический детектив по мотивам цикла Виктора Дашкевича о графе Аверине  
**Формат:** демонстрационный блокнот с полным разбором задания  
**Цель:** показать, как развить проект из задания №1, добавив абстрактные классы, протоколы, `dataclass`-конфигурации и миксин логирования

---

## 📖 Контекст: что уже есть в проекте

В практической работе №1 мы создали процедурный прототип и провели его рефакторинг в ООП. В проекте уже есть:

| Класс | Ответственность |
|---|---|
| `Character` | Базовый персонаж (имя, описание) |
| `Detective` | Граф Аверин: энергия, улики, Плеть |
| `Div` | Кузя: формы, лояльность, разведка |
| `Location` | Локация: улики, персонажи |
| `Clue` | Улика: название, описание, важность |
| `Game` | Игровой цикл |
| `UI` | Ввод/вывод |

## 🚧 Какие проблемы остались после задания №1

Первое задание дало нам работающую ООП-версию игры, но оставило несколько архитектурных долгов, которые станут критичными при развитии проекта:

**Проблема 1. Нет единого базового класса для игровых объектов.**  
`Detective`, `Div`, `Location` — все они «игровые сущности», но общего предка нет. Мы не можем написать код, который работает с любой сущностью единообразно. Игровой цикл вынужден знать о каждом классе отдельно.

**Проблема 2. Способности не выделены в интерфейсы.**  
Понятия «двигаться», «отрисовываться», «сталкиваться» существуют неявно. Мы не можем проверить, что объект умеет рисоваться, не зная его конкретный класс. Нельзя передать «список всего, что можно нарисовать» — только список конкретных объектов.

**Проблема 3. Конфигурация задана магическими числами.**  
В коде `Detective` зашиты значения `energy=100`, `cost=10`, `cost=20`. Их нельзя изменить без правки кода. Тестировать разные сценарии сложно. Значения дублируются между классами.

**Проблема 4. Нет логирования.**  
Когда что-то идёт не так (детектив теряет энергию необъяснимо, улика не добавляется), мы не можем посмотреть, что произошло. Отладка сводится к расстановке `print()` вручную.

## 🎯 Что демонстрирует этот блокнот и как это улучшит проект

| Инструмент | Проблема, которую решает | Как улучшает проект |
|---|---|---|
| **ABC `GameObject`** | Нет единого контракта для сущностей | Игровой цикл работает с любым `GameObject` единообразно |
| **Протоколы `Movable`/`Drawable`/`Collidable`** | Способности неявны | Рендерер может принимать «всё, что умеет `draw()`» |
| **`@dataclass` конфиги** | Магические числа | Настройки вынесены в одно место, легко менять |
| **`frozen=True`** | Конфиг можно случайно изменить | Гарантия неизменности во время игры |
| **Миксин `LoggingMixin`** | Нет истории действий | Логирование подключается одной строкой |

В конце блокнота проект будет готов к следующим модулям: SOLID, паттернам, ECS.

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# 1. Абстрактный класс `GameObject`

## Зачем этот шаг

### Контекст в проекте

Сейчас в проекте есть три класса, которые по сути являются игровыми объектами: `Detective`, `Div` (в будущем), `Location`. Но у них нет общего предка. Это создаёт три конкретные проблемы:

1. **Игровой цикл вынужден различать типы.** В `Game.run()` мы вручную вызываем методы конкретных объектов. Если завтра появятся `Item`, `Trap`, `Door`, придётся добавлять ветки в игровой цикл.
2. **Нет гарантии, что объект умеет обновляться.** Если программист забудет реализовать `update()` в новом классе, ошибка проявится только во время выполнения — и, скорее всего, в самый неподходящий момент.
3. **Дублирование полей.** У каждого класса своя логика для `id`, `name`, `active`. При добавлении нового класса приходится копировать эту логику.

### Что даёт ABC

**Абстрактный базовый класс** решает все три проблемы:

- **Единый контракт.** `GameObject` объявляет: «любой игровой объект умеет `update()` и `draw()`». Игровой цикл работает с `GameObject`, а не с конкретными классами.
- **Гарантия реализации.** Если подкласс забудет реализовать `update()`, Python выбросит `TypeError` **при попытке создать экземпляр**. Ошибка обнаруживается сразу, а не через час игры.
- **Общие поля.** `id`, `name`, `active` определяются один раз в базовом классе. Подклассы наследуют их автоматически.

### Как это улучшит проект

**До ABC:**
```python
# Игровой цикл знает о каждом классе
def update_all(self, dt):
    self.detective.update_state(dt)
    for div in self.divs:
        div.tick(dt)
    for loc in self.locations:
        loc.refresh(dt)
```

**После ABC:**
```python
# Игровой цикл работает с любым GameObject
def update_all(self, objects: list[GameObject], dt: float):
    for obj in objects:
        if obj.active:
            obj.update(dt)
```

Второй вариант короче, расширяем и не требует правок при добавлении новых типов объектов.

### Как работает ABC технически

Модуль `abc` предоставляет:
- `ABC` — базовый класс для создания ABC. Все классы, наследующие `ABC`, становятся абстрактными.
- `@abstractmethod` — декоратор для методов, которые **обязаны** быть переопределены в подклассе.

Механизм работает через метакласс `ABCMeta`, который при создании экземпляра проверяет, что все абстрактные методы реализованы. Если нет — `TypeError`.

**Важное отличие от Java/C++:** в Python ABC не «компилирует» контракт, а проверяет его во время выполнения. Но проверка происходит **при создании объекта**, а не при вызове метода — это ранняя диагностика.

### Для углублённого изучения

- [Real Python: Implementing Interfaces in Python: ABCs and Protocols](https://realpython.com/python-interface/) — подробный разбор ABC с примерами и сравнением с протоколами.
- [Python docs: abc — Abstract Base Classes](https://docs.python.org/3/library/abc.html) — официальная документация с примерами использования.
- [PEP 3119 — Introducing Abstract Base Classes](https://peps.python.org/pep-3119/) — исторический документ, объясняющий мотивацию ABC.
- Лутц М. «Изучаем Python», глава 26 — объектная модель Python и абстрактные классы.

In [ ]:
%%writefile mystic_detective/core/game_object.py
"""Абстрактный базовый класс для всех игровых сущностей."""

from abc import ABC, abstractmethod


class GameObject(ABC):
    """Базовый класс для всего, что существует в игровом мире.

    Контракт:
    - каждый игровой объект имеет уникальный id;
    - каждый игровой объект может быть активным или неактивным;
    - каждый игровой объект обязан уметь обновляться (update);
    - каждый игровой объект обязан уметь отрисовываться (draw).

    Эти гарантии позволяют игровому циклу работать
    с любыми объектами единообразно.
    """

    _next_id: int = 0

    def __init__(self, name: str):
        self._id = GameObject._next_id
        GameObject._next_id += 1
        self._name = name
        self._active = True

    @property
    def id(self) -> int:
        return self._id

    @property
    def name(self) -> str:
        return self._name

    @property
    def active(self) -> bool:
        return self._active

    def deactivate(self) -> None:
        """Пометить объект как неактивный (удалить из игры).

        В играх объекты часто не удаляют физически — это дорого.
        Вместо этого их помечают как неактивные, а игровой цикл
        пропускает их при update и draw.
        """
        self._active = False

    @abstractmethod
    def update(self, dt: float) -> None:
        """Обновить состояние объекта за время dt (секунды).

        Вызывается каждый кадр игрового цикла.
        """
        ...

    @abstractmethod
    def draw(self) -> str:
        """Вернуть строковое представление для отрисовки.

        В текстовой игре draw возвращает строку для вывода.
        В Pygame-версии draw будет рисовать спрайт на экране.
        """
        ...

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(id={self._id}, name={self._name!r})"


### Проверка: ABC нельзя инстанцировать

Первое, что мы проверяем — что сам `GameObject` нельзя создать. Это ключевое свойство ABC: он существует только как контракт. Попытка `GameObject("test")` должна выбросить `TypeError`.

**Почему это важно:** если бы ABC можно было инстанцировать, разработчик мог бы случайно создать «пустой» объект и передать его в игровой цикл. Игровой цикл вызвал бы `update()` и `draw()` — а они не реализованы. Результат: непредсказуемое поведение или AttributeError в глубине игры.

In [ ]:
import sys
sys.path.insert(0, 'mystic_detective')

from core.game_object import GameObject

try:
    GameObject("test")
except TypeError as e:
    print(f"✅ Ожидаемая ошибка: {e}")

---
# 2. Протоколы `Movable`, `Drawable`, `Collidable`

## Зачем этот шаг

### Контекст в проекте

В игре есть объекты, которые **умеют** разные вещи:
- Детектив может двигаться и рисоваться, но не сталкивается (пока).
- Див Кузя может двигаться, рисоваться и быть невидимым для врагов.
- В будущем появятся двери (сталкиваются), предметы (можно подобрать), снаряды Плети (летят и сталкиваются).

Эти способности — не типы объектов, а **роли**. Детектив не «является Movable», он «играет роль Movable». Протоколы позволяют описать эти роли без жёсткой иерархии наследования.

### Проблема, которую решают протоколы

**Без протоколов** нам пришлось бы либо:
1. **Наследоваться от ABC.** Но тогда `Detective` должен наследоваться от `Movable`, `Drawable`, `Collidable` — а он не всегда всё это умеет. Получим «толстые» базовые классы и комбинаторный взрыв подклассов (см. модуль 1).
2. **Проверять наличие методов вручную.** `if hasattr(obj, 'draw'): obj.draw()` — работает, но не читается и не проверяется статически.

**С протоколами** мы пишем:
```python
def render_all(objects: list[Drawable], screen):
    for obj in objects:
        obj.draw()
```
И это работает для любого объекта с методом `draw()`. Плюс mypy и IDE проведут статическую проверку.

### Как это улучшит проект

**До протоколов:**
```python
# Рендерер зависит от конкретных классов
def render(self, detective: Detective, divs: list[Div], locations: list[Location]):
    detective.draw()
    for div in divs:
        div.draw()
    for loc in locations:
        loc.draw()
```

**После протоколов:**
```python
# Рендерер работает с абстракцией
def render(self, objects: list[Drawable]):
    for obj in sorted(objects, key=lambda o: o.z_order):
        obj.draw()
```

Разница принципиальная: второй вариант **не знает о конкретных классах**. Добавление нового типа отрисовываемого объекта не требует правки рендерера.

### Протоколы vs ABC: когда что использовать

| Ситуация | Что использовать |
|---|---|
| Вы владеете иерархией и хотите контролировать её | **ABC** |
| Важна способность («что умеет»), а не тип («чем является») | **Protocol** |
| Внешние объекты, которые вы не контролируете | **Protocol** |
| Базовый класс с общей реализацией | **ABC** |

В нашем проекте это означает:
- `GameObject` — ABC, потому что это базовый класс с общим кодом (`id`, `active`).
- `Movable`, `Drawable`, `Collidable` — протоколы, потому что это способности, а не типы.

### Почему `@runtime_checkable`

По умолчанию протоколы работают только на уровне статической типизации (mypy, IDE). Во время выполнения `isinstance(obj, MyProtocol)` выбросит ошибку. Декоратор `@runtime_checkable` включает проверку во время выполнения.

**Важно:** `runtime_checkable` проверяет только **наличие** методов, но не их сигнатуры. Если метод `draw` ожидает три аргумента, а объект реализует его без аргументов — проверка пройдёт, но вызов упадёт. Это компромисс: полная проверка сигнатур требует слишком много ресурсов.

### Для углублённого изучения

- [PEP 544 — Protocols: Structural subtyping](https://peps.python.org/pep-0544/) — оригинальный PEP, объясняющий мотивацию.
- [Real Python: Structural Subtyping with Protocols](https://realpython.com/python-protocol/) — практический разбор с примерами.
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html) — спецификация протоколов.
- Ramalho L. «Fluent Python», 2-е изд., глава 13 — протоколы, интерфейсы и абстракции.

In [ ]:
%%writefile mystic_detective/core/protocols.py
"""Протоколы способностей игровых объектов.

Каждый протокол описывает одну роль, которую может играть объект.
Объект может реализовывать несколько протоколов одновременно.
"""

from typing import Protocol, runtime_checkable


@runtime_checkable
class Movable(Protocol):
    """Роль: объект умеет перемещаться в пространстве.

    Реализуется персонажами, снарядами, движущимися предметами.
    """

    def move(self, dx: float, dy: float) -> None:
        """Переместиться на (dx, dy)."""
        ...


@runtime_checkable
class Drawable(Protocol):
    """Роль: объект можно отрисовать.

    Реализуется всем, что должно появляться на экране:
    персонажи, локации, UI-элементы, эффекты.
    """

    def draw(self) -> str:
        """Вернуть строковое представление для отрисовки."""
        ...


@runtime_checkable
class Collidable(Protocol):
    """Роль: объект может участвовать в коллизиях.

    Реализуется стенами, дверями, снарядами, врагами.
    Не все объекты должны сталкиваться: например, звук или UI — нет.
    """

    @property
    def collision_radius(self) -> float:
        """Радиус для проверки коллизий."""
        ...

    def collides_with(self, other: "Collidable") -> bool:
        """Проверить коллизию с другим объектом."""
        ...


### Проверка протоколов

Мы создаём простой класс `SimpleObject` с методами `move` и `draw` и проверяем, что он автоматически совместим с протоколами `Movable` и `Drawable` — без наследования. При этом `Collidable` он не удовлетворяет, потому что нет `collision_radius` и `collides_with`.

**Это ключевая демонстрация структурной типизации:** важна «форма» объекта, а не его происхождение.

In [ ]:
from core.protocols import Movable, Drawable, Collidable


class SimpleObject:
    def move(self, dx, dy):
        print(f"Moved by ({dx}, {dy})")

    def draw(self):
        return "SimpleObject"


obj = SimpleObject()
print(f"obj is Movable: {isinstance(obj, Movable)}")
print(f"obj is Drawable: {isinstance(obj, Drawable)}")
print(f"obj is Collidable: {isinstance(obj, Collidable)}")
print(f"int is Drawable: {isinstance(42, Drawable)}")

---
# 3. `@dataclass` для конфигураций и компонентов

## Зачем этот шаг

### Контекст в проекте

В коде из задания №1 настройки игры **зашиты в логику**:

```python
class Detective(Character):
    def __init__(self, name, energy=100):
        self._energy = energy
        self._max_energy = energy
        # ...

    def investigate(self, location_name):
        if self._energy < 10:  # магическое число
            return "Не хватает энергии"
        self._energy -= 10  # дублируется

    def use_whip(self):
        if self._energy < 20:  # другое магическое число
            ...
        self._energy -= 20
```

**Три проблемы:**
1. **Магические числа.** Значения `10` и `20` разбросаны по коду. Чтобы понять баланс, нужно читать все методы.
2. **Невозможно изменить без правки кода.** Хотите сделать игру легче — правьте `detective.py`. Хотите другую сложность для теста — снова правьте.
3. **Дублирование.** Одно и то же число встречается в проверке и в вычитании. Легко рассогласовать при рефакторинге.

### Что даёт `@dataclass`

**`@dataclass`** автоматически генерирует `__init__`, `__repr__`, `__eq__` на основе аннотаций типов. Это устраняет шаблонный код для классов-данных.

**`frozen=True`** делает объект неизменяемым: после создания нельзя изменить ни одно поле. Это идеально для конфигураций — их не должно менять во время игры. Дополнительный бонус: замороженные dataclass **хешируемы**, их можно использовать как ключи словарей.

**`field(default_factory=...)`** решает проблему изменяемых значений по умолчанию. Без него `items: list = []` привёл бы к тому, что все объекты делили бы один список (проблема из модуля 1).

### Как это улучшит проект

**До dataclass:**
```python
detective = Detective("Аверин", energy=100)
detective.investigate("особняк")  # минус 10 (магическое число)
detective.use_whip()               # минус 20 (магическое число)
```

**После dataclass:**
```python
cfg = GameConfig()  # все настройки в одном месте
detective = Detective(config=DetectiveConfig(), game_config=cfg)
detective.investigate("особняк")  # минус cfg.investigate_cost
detective.use_whip()               # минус cfg.whip_cost
```

Теперь для изменения баланса достаточно создать другой `GameConfig`:
```python
easy_mode = GameConfig(investigate_cost=5, whip_cost=10)
hard_mode = GameConfig(investigate_cost=20, whip_cost=40)
```

### Почему `frozen=True` для конфигов

Конфигурация — это **иммутабельные данные**. Игра не должна менять `GameConfig` во время выполнения. Если бы `frozen=True` не было, код мог бы случайно сделать так:
```python
cfg.max_energy = -1  # случайная ошибка, игра сломана
```
С `frozen=True` такая ошибка обнаруживается сразу — при попытке присваивания.

**Обратите внимание:** `frozen=True` делает неизменяемым только сам объект, но не вложенные объекты. Если бы `LocationConfig.clues` был списком, его можно было бы изменить. Поэтому для списков используется `field(default_factory=list)` и допускается мутация содержимого.

### Когда НЕ использовать dataclass

- Если у класса есть сложная логика инициализации (валидация, вычисления).
- Если нужно наследование с переопределением `__init__` в сложной форме.
- Если объект должен иметь нестандартные `__init__` или `__eq__`.

Для конфигураций, компонентов, DTO, событий — dataclass идеален.

### Для углублённого изучения

- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html) — полная документация с примерами.
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/) — мотивация и дизайн.
- [Real Python: Data Classes in Python 3.7+](https://realpython.com/python-data-classes/) — практический туториал.

In [ ]:
%%writefile mystic_detective/core/config.py
"""Конфигурации игры и персонажей.

Все настройки вынесены в dataclass-классы:
- GameConfig: общие параметры (стоимость действий, лимиты)
- DetectiveConfig: параметры детектива
- DivConfig: параметры дива
- LocationConfig: конфигурация локации
- ClueData: данные улики (компонент)

Использование:
    cfg = GameConfig()
    easy = GameConfig(investigate_cost=5)
    hard = GameConfig(investigate_cost=20)
"""

from dataclasses import dataclass, field


@dataclass(frozen=True)
class GameConfig:
    """Общие настройки игры.

    frozen=True: настройки нельзя изменить после создания.
    Это защищает от случайных изменений во время игры.
    """

    title: str = "Тайна исчезнувшего дива"
    max_energy: int = 100
    investigate_cost: int = 10
    whip_cost: int = 20
    rest_amount: int = 5


@dataclass(frozen=True)
class DetectiveConfig:
    """Параметры детектива."""

    name: str = "Граф Гермес Аверин"
    energy: int = 100
    max_energy: int = 100


@dataclass(frozen=True)
class DivConfig:
    """Параметры дива-напарника."""

    name: str = "Кузя"
    form: str = "кот"
    loyalty: int = 50
    forms: tuple = ("кот", "юноша", "ворон")


@dataclass
class LocationConfig:
    """Конфигурация локации.

    field(default_factory=list) создаёт новый список
    для каждого экземпляра — иначе все локации делили бы один список.
    """

    name: str
    description: str = ""
    clues: list = field(default_factory=list)
    characters: list = field(default_factory=list)


@dataclass
class ClueData:
    """Данные улики (компонент)."""

    name: str
    description: str = ""
    is_key: bool = False


### Проверка dataclass

Мы проверяем три свойства dataclass:
1. **Автоматический `__repr__`** — печать показывает все поля.
2. **Автоматический `__eq__`** — два конфига с одинаковыми полями равны.
3. **`frozen=True` работает** — попытка изменить поле выбрасывает `FrozenInstanceError`.

In [ ]:
from core.config import GameConfig, DetectiveConfig, DivConfig, ClueData

cfg = GameConfig()
print(cfg)

det = DetectiveConfig()
print(det)

div = DivConfig()
print(div)

clue = ClueData("Записка", "План Пустоши", is_key=True)
print(clue)

# frozen: попытка изменить вызовет ошибку
try:
    cfg.max_energy = 200
except Exception as e:
    print(f"✅ Ожидаемая ошибка: {type(e).__name__}: {e}")

# Проверка независимости списков
loc_a = LocationConfig("особняк")
loc_b = LocationConfig("академия")
loc_a.clues.append("Странный запах гари")
print(f"loc_a.clues: {loc_a.clues}")
print(f"loc_b.clues: {loc_b.clues}")

---
# 4. Миксин `LoggingMixin`

## Зачем этот шаг

### Контекст в проекте

В игре происходит много событий: детектив осматривает локацию, теряет энергию, находит улику, див меняет форму. Когда что-то идёт не так (энергия уходит не туда, улика не добавляется), разработчик хочет понять **что произошло и в каком порядке**.

Без логирования единственный инструмент — `print()`. Но:
- `print()` нельзя выключить, не удаляя код;
- `print()` не имеет уровней (info/warning/error);
- `print()` не имеет метки времени и имени класса;
- `print()` невозможно перенаправить в файл.

### Что даёт миксин

**`LoggingMixin`** добавляет любому классу методы `log()`, `log_warning()`, `log_error()` через стандартный модуль `logging`. Это:
- **стандартно** — использует официальный `logging`;
- **гибко** — можно выключить или перенаправить;
- **переиспользуемо** — подключается одной строкой наследования.

### Почему именно миксин

Логирование — классический пример **сквозной функциональности** (cross-cutting concern). Она нужна многим классам, но не относится ни к одной предметной области. Варианты реализации:

| Подход | Плюсы | Минусы |
|---|---|---|
| Глобальная функция `log()` | Просто | Загрязняет namespace, не привязана к классу |
| Базовый класс `LoggedObject` | Наследование | Занимает единственный слот наследования |
| **Миксин** | Гибко, комбинируется | Требует понимания MRO |

Миксин выигрывает, потому что не мешает основной иерархии: `Detective` уже наследуется от `GameObject`, но может дополнительно примешать `LoggingMixin`.

### Как это улучшит проект

**До миксина:**
```python
def investigate(self, location):
    print(f"Investigate {location}")  # отладочный print
    self._energy -= 10
    print(f"Energy now {self._energy}")
```

**После миксина:**
```python
def investigate(self, location):
    self.log(f"Осмотр локации {location}")
    self._energy -= self._game_config.investigate_cost
    self.log(f"Энергия: {self._energy}")
```

Плюс: в консоли видно `[Detective] INFO: Осмотр локации особняк` — понятно, кто и что сделал. В продакшене логи можно отключить или писать в файл.

### Правила создания миксинов

1. **Не иметь `__init__`** — миксин не должен требовать инициализации.
2. **Не иметь состояния.** Только методы. Если нужны пустые слоты — `__slots__ = ()`.
3. **Вызывать `super()`** — для кооперативного наследования (в нашем миксине не требуется, но помните правило).
4. **Имя оканчивается на `Mixin`** — соглашение, облегчающее чтение.
5. **Быть независимым** — миксин не должен предполагать, с каким классом его смешают.

### Порядок наследования

Когда вы пишете `class Detective(LoggingMixin, GameObject)`, MRO строится так: `Detective → LoggingMixin → GameObject → ABC → object`. Методы `LoggingMixin` имеют приоритет над `GameObject`. Если оба класса определяют метод с одним именем, победит `LoggingMixin`.

**Правило:** миксин всегда ставится **первым** в списке наследования.

### Для углублённого изучения

- [Python Cookbook: Extending Classes with Mixins](https://python-cookbook.readthedocs.io/zh-cn/stable/c08/p18_extending_classes_with_mixins.html) — классические рецепты.
- [Real Python: Multiple Inheritance and Mixins](https://realpython.com/inheritance-composition-python/) — композиция и миксины.
- [Python docs: logging](https://docs.python.org/3/library/logging.html) — стандартный модуль логирования.
- [Real Python: Logging in Python](https://realpython.com/python-logging/) — практический туториал.

In [ ]:
%%writefile mystic_detective/core/mixins.py
"""Миксины для переиспользования поведения.

Миксин — это класс без состояния, добавляющий функциональность
другим классам через множественное наследование.

Правила миксина:
- нет __init__;
- нет атрибутов экземпляра;
- только методы;
- имя оканчивается на Mixin.
"""

import logging


class LoggingMixin:
    """Миксин: добавляет логирование действий объекта.

    Использование:
        class Detective(LoggingMixin, GameObject):
            def investigate(self, location):
                self.log(f"Осмотр {location}")

    Логи выводятся в стандартный поток с меткой класса.
    Уровень логирования INFO по умолчанию.
    """

    __slots__ = ()

    def log(self, message: str) -> None:
        """Записать информационное сообщение."""
        logger = logging.getLogger(self.__class__.__name__)
        if not logger.handlers:
            handler = logging.StreamHandler()
            formatter = logging.Formatter(
                "[%(name)s] %(levelname)s: %(message)s"
            )
            handler.setFormatter(formatter)
            logger.addHandler(handler)
            logger.setLevel(logging.INFO)
        logger.info(message)

    def log_warning(self, message: str) -> None:
        """Записать предупреждение."""
        logging.getLogger(self.__class__.__name__).warning(message)

    def log_error(self, message: str) -> None:
        """Записать ошибку."""
        logging.getLogger(self.__class__.__name__).error(message)


### Проверка миксина

Мы создаём класс `TestClass`, который наследует только `LoggingMixin`, и вызываем `log()`. В консоли должно появиться сообщение с меткой `[TestClass] INFO: ...`.

Обратите внимание: в миксине нет `__init__` — он работает с любым классом. Это и есть суть миксина.

In [ ]:
from core.mixins import LoggingMixin


class TestClass(LoggingMixin):
    def do_work(self):
        self.log("Working...")


obj = TestClass()
obj.do_work()
print("✅ Логирование работает")

---
# 5. Интеграция: обновление `Detective`

## Зачем этот шаг

### Контекст в проекте

Всё, что мы создали в предыдущих разделах (`GameObject`, протоколы, dataclass, миксин) — это отдельные инструменты. Теперь нужно показать, как они **работают вместе** на реальном классе проекта.

Обновлённый `Detective`:
- **наследует `GameObject`** — теперь он часть единой иерархии сущностей;
- **наследует `LoggingMixin`** — все действия логируются;
- **реализует `Drawable`** через метод `draw()`;
- **использует `DetectiveConfig`** и `GameConfig` вместо магических чисел;
- **гарантированно реализует `update()`** — иначе Python не даст создать экземпляр.

### Как это улучшит проект

**До интеграции:**
```python
detective = Detective("Аверин")  # без логирования, без id, без контракта
detective.investigate("особняк")  # магическое число 10
# Никто не может проверить, что detective.active или detective.id
```

**После интеграции:**
```python
detective = Detective(
    config=DetectiveConfig(),
    game_config=GameConfig(),
)
detective.investigate("особняк")
# [Detective] INFO: Осмотр локации особняк (энергия: 90)
assert isinstance(detective, GameObject)
assert isinstance(detective, Drawable)
assert detective.active
print(detective.id)  # уникальный id
```

Теперь детектив:
- имеет **уникальный id** — можно отслеживать в коллекциях;
- помечается как **неактивный** через `deactivate()`;
- **логирует** каждое действие;
- проходит **проверку `isinstance`** для `Drawable`;
- **читает настройки** из конфига.

### Порядок наследования: почему миксин первым

```python
class Detective(LoggingMixin, GameObject):
```

MRO строится так: `Detective → LoggingMixin → GameObject → ABC → object`. Если бы мы поставили `GameObject` первым, методы `LoggingMixin` могли бы не вызываться при конфликтах. Правило: **миксин всегда первый**.

### Конфигурация: два уровня

Мы используем два конфига:
- **`DetectiveConfig`** — параметры самого детектива (имя, начальная энергия);
- **`GameConfig`** — общие параметры игры (стоимость действий, лимиты).

Это разделение ответственности: если завтра появится `DivConfig` с параметрами дива, оно не смешается с общими настройками.

### Для углублённого изучения

- [Real Python: Method Resolution Order](https://realpython.com/python-mro/) — как Python определяет порядок вызовов.
- [Python docs: MRO](https://docs.python.org/3/howto/mro.html) — официальный HowTo.
- [Real Python: Supercharge Your Classes With Python super()](https://realpython.com/python-super/) — работа с `super()`.

In [ ]:
%%writefile mystic_detective/entities/detective_v2.py
"""Обновлённый Detective: наследует GameObject и LoggingMixin,
использует dataclass-конфиги вместо магических чисел.

Демонстрирует интеграцию всех инструментов модуля 2.
"""

from core.game_object import GameObject
from core.mixins import LoggingMixin
from core.config import DetectiveConfig, GameConfig
from entities.clue import Clue
from entities.div import Div


class Detective(LoggingMixin, GameObject):
    """Граф Аверин: колдун, детектив, владелец Плети.

    Наследует:
    - LoggingMixin — методы log(), log_warning(), log_error()
    - GameObject — контракт update()/draw() и общие поля id/name/active

    Реализует протокол Drawable через метод draw().
    """

    def __init__(
        self,
        config: DetectiveConfig | None = None,
        game_config: GameConfig | None = None,
        div: Div | None = None,
    ):
        cfg = config if config is not None else DetectiveConfig()
        self._game_config = game_config if game_config is not None else GameConfig()
        super().__init__(cfg.name)
        self._energy = cfg.energy
        self._max_energy = cfg.max_energy
        self._clues: list[Clue] = []
        self._div = div
        self.log(f"Создан детектив {cfg.name}")

    @property
    def energy(self) -> int:
        return self._energy

    @property
    def max_energy(self) -> int:
        return self._max_energy

    @property
    def clues(self) -> list[Clue]:
        return list(self._clues)

    @property
    def div(self) -> Div | None:
        return self._div

    def add_clue(self, clue: Clue) -> None:
        """Добавить улику. Дубликаты игнорируются."""
        if clue not in self._clues:
            self._clues.append(clue)
            self.log(f"Найдена улика: {clue.name}")

    def investigate(self, location_name: str) -> str:
        """Осмотреть локацию. Стоимость берётся из GameConfig."""
        cost = self._game_config.investigate_cost
        if self._energy < cost:
            self.log_warning(f"Не хватает энергии: нужно {cost}, есть {self._energy}")
            return "Не хватает энергии для осмотра."
        self._energy -= cost
        self.log(f"Осмотр локации {location_name} (энергия: {self._energy})")
        return f"{self._name} осматривает {location_name}"

    def use_whip(self) -> str:
        """Применить Плеть. Стоимость берётся из GameConfig."""
        cost = self._game_config.whip_cost
        if self._energy < cost:
            self.log_warning(f"Не хватает энергии для Плети: нужно {cost}")
            return "Не хватает энергии для Плети."
        self._energy -= cost
        self.log(f"Применение Плети (энергия: {self._energy})")
        return "Плеть рассекает воздух!"

    def rest(self, amount: int | None = None) -> None:
        """Отдохнуть. По умолчанию восстанавливает rest_amount из GameConfig."""
        amt = amount if amount is not None else self._game_config.rest_amount
        self._energy = min(self._max_energy, self._energy + amt)
        self.log(f"Отдых: энергия восстановлена до {self._energy}")

    def update(self, dt: float) -> None:
        """Обновление состояния (реализация абстрактного метода GameObject).

        Пока ничего не делает: детектив управляется игроком.
        Но контракт выполнен — иначе нельзя создать экземпляр.
        """
        pass

    def draw(self) -> str:
        """Отрисовка (реализация абстрактного метода GameObject)."""
        return f"[Detective] {self._name} (HP: {self._energy}/{self._max_energy})"


### Проверка интеграции

Мы создаём детектива с конфигами, проверяем:
1. Логирование работает — в консоли видны сообщения `[Detective]`.
2. `isinstance(det, Drawable)` возвращает `True` — протокол соблюдён.
3. `investigate`, `use_whip`, `rest` используют значения из `GameConfig`.
4. `draw()` возвращает читаемое представление.

In [ ]:
from entities.div import Div
from entities.detective_v2 import Detective
from core.config import DetectiveConfig, GameConfig
from core.protocols import Drawable
from core.game_object import GameObject

div = Div("Кузя", form="кот", loyalty=50)
det = Detective(
    config=DetectiveConfig(),
    game_config=GameConfig(),
    div=div,
)

print(det.draw())
print(f"isinstance(det, Drawable): {isinstance(det, Drawable)}")
print(f"isinstance(det, GameObject): {isinstance(det, GameObject)}")
print(f"Уникальный id: {det.id}")

det.investigate("особняк")
det.use_whip()
det.rest()
print(det.draw())

# Проверка на лёгкой сложности
easy_cfg = GameConfig(investigate_cost=5, whip_cost=10)
det_easy = Detective(game_config=easy_cfg)
det_easy.investigate("академия")
print(det_easy.draw())

---
# 6. Юнит-тесты

## Зачем этот шаг

### Контекст в проекте

Мы добавили четыре новых инструмента: ABC, протоколы, dataclass, миксин. Каждый из них имеет **контракт**, который нужно проверить:
- ABC **обязан** выбрасывать `TypeError` при попытке создать экземпляр без реализации методов.
- Протоколы **обязаны** корректно определять совместимость через `isinstance`.
- `frozen=True` **обязан** запрещать изменение полей.
- Миксин **обязан** работать в любом классе без `__init__`.

Без тестов эти контракты — просто обещания. Тесты превращают их в гарантии.

### Почему важно тестировать именно это

В игровом проекте регрессии в базовых классах катастрофичны. Если кто-то случайно переопределит `__init__` в `GameObject` так, что ABC перестанет работать, — все игровые объекты потеряют контракт. Тесты на ABC поймают это мгновенно.

Аналогично с dataclass: если кто-то уберёт `frozen=True`, конфиги станут изменяемыми, и случайное изменение в одном месте игры сломает баланс везде. Тест `test_game_config_frozen` защитит от этого.

### Что мы тестируем

| Файл тестов | Что проверяет |
|---|---|
| `test_game_object.py` | ABC нельзя инстанцировать; подкласс без методов — тоже |
| `test_protocols.py` | `isinstance` работает по структуре, а не по наследованию |
| `test_config.py` | `frozen=True` запрещает изменения; списки независимы |
| `test_mixins.py` | `LoggingMixin` работает в любом классе |

### Для углублённого изучения

- [pytest documentation](https://docs.pytest.org/) — официальная документация.
- [Real Python: Testing Your Code with pytest](https://realpython.com/pytest-python-testing/) — практический туториал.
- [Python docs: unittest.mock](https://docs.python.org/3/library/unittest.mock.html) — Mock для изоляции.

In [ ]:
%%writefile mystic_detective/tests/test_game_object.py
"""Тесты для ABC GameObject.

Проверяют ключевой контракт ABC:
- нельзя инстанцировать абстрактный класс;
- нельзя инстанцировать подкласс без реализации абстрактных методов;
- полный подкласс работает и получает уникальный id.
"""

import pytest
from core.game_object import GameObject


class TestGameObjectABC:
    def test_cannot_instantiate_abstract(self):
        """ABC нельзя создать напрямую."""
        with pytest.raises(TypeError):
            GameObject("test")

    def test_subclass_must_implement_methods(self):
        """Подкласс без update/draw нельзя создать."""
        class Incomplete(GameObject):
            pass

        with pytest.raises(TypeError):
            Incomplete("test")

    def test_concrete_subclass_works(self):
        """Полный подкласс создаётся и имеет уникальный id."""
        class Concrete(GameObject):
            def update(self, dt):
                pass

            def draw(self):
                return "Concrete"

        a = Concrete("a")
        b = Concrete("b")
        assert a.name == "a"
        assert isinstance(a.id, int)
        assert a.active
        assert a.id != b.id  # уникальные id

In [ ]:
%%writefile mystic_detective/tests/test_protocols.py
"""Тесты для протоколов.

Проверяют структурную типизацию:
объект совместим с протоколом по наличию методов,
а не по наследованию.
"""

from core.protocols import Movable, Drawable, Collidable


class TestProtocols:
    def test_object_with_draw_is_drawable(self):
        class WithDraw:
            def draw(self):
                return "x"

        assert isinstance(WithDraw(), Drawable)

    def test_object_without_draw_is_not_drawable(self):
        class WithoutDraw:
            pass

        assert not isinstance(WithoutDraw(), Drawable)

    def test_object_with_move_is_movable(self):
        class WithMove:
            def move(self, dx, dy):
                pass

        assert isinstance(WithMove(), Movable)

    def test_object_with_collision_is_collidable(self):
        class WithCollision:
            @property
            def collision_radius(self):
                return 1.0

            def collides_with(self, other):
                return False

        assert isinstance(WithCollision(), Collidable)

    def test_int_is_not_drawable(self):
        assert not isinstance(42, Drawable)


In [ ]:
%%writefile mystic_detective/tests/test_config.py
"""Тесты для dataclass-конфигураций.

Проверяют:
- frozen=True запрещает изменение полей;
- значения по умолчанию корректны;
- default_factory создаёт независимые списки.
"""

import pytest
from core.config import GameConfig, DetectiveConfig, LocationConfig, ClueData


class TestConfig:
    def test_game_config_frozen(self):
        cfg = GameConfig()
        with pytest.raises(Exception):
            cfg.max_energy = 200

    def test_game_config_defaults(self):
        cfg = GameConfig()
        assert cfg.max_energy == 100
        assert cfg.investigate_cost == 10
        assert cfg.whip_cost == 20

    def test_game_config_equality(self):
        a = GameConfig()
        b = GameConfig()
        assert a == b

    def test_location_config_independent_lists(self):
        a = LocationConfig("особняк")
        b = LocationConfig("академия")
        a.clues.append("улика")
        assert a.clues == ["улика"]
        assert b.clues == []

    def test_clue_data(self):
        clue = ClueData("Записка", "План", is_key=True)
        assert clue.name == "Записка"
        assert clue.is_key


In [ ]:
%%writefile mystic_detective/tests/test_mixins.py
"""Тесты для LoggingMixin.

Проверяют, что миксин работает в любом классе
без __init__ и не требует дополнительной инициализации.
"""

from core.mixins import LoggingMixin


class TestLoggingMixin:
    def test_log_does_not_crash(self):
        class Foo(LoggingMixin):
            pass

        foo = Foo()
        foo.log("test message")

    def test_log_warning(self):
        class Foo(LoggingMixin):
            pass

        foo = Foo()
        foo.log_warning("warning")

    def test_log_error(self):
        class Foo(LoggingMixin):
            pass

        foo = Foo()
        foo.log_error("error")

    def test_mixin_has_no_state(self):
        """Миксин не имеет собственного состояния."""
        assert LoggingMixin.__slots__ == ()


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-v', '--tb=short'],
    cwd='mystic_detective',
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Код выхода: {result.returncode}")

---
# 7. Итоги и литература

## Что мы сделали и зачем

| Инструмент | Зачем добавили | Что улучшилось в проекте |
|---|---|---|
| **ABC `GameObject`** | Не было единого контракта | Игровой цикл работает с любой сущностью единообразно |
| **Protocol `Movable`** | Способности были неявными | Рендерер и физика работают с абстракциями |
| **Protocol `Drawable`** | То же | Любой объект с `draw()` автоматически отрисовывается |
| **Protocol `Collidable`** | То же | Коллизии выделены в явный контракт |
| **`@dataclass`** | Магические числа | Все настройки в одном месте, легко менять |
| **`frozen=True`** | Конфиг мог быть случайно изменён | Гарантия неизменности во время игры |
| **`field(default_factory)`** | Изменяемые defaults | Независимые списки у каждого объекта |
| **`LoggingMixin`** | Не было истории действий | Логирование подключается одной строкой |

## Что изменилось в проекте

**До задания №2:**
- `Detective` — самостоятельный класс без общего предка;
- настройки зашиты в код;
- нет логирования;
- способности неявны.

**После задания №2:**
- `Detective` наследует `GameObject` и `LoggingMixin`;
- настройки вынесены в `GameConfig` и `DetectiveConfig`;
- каждое действие логируется;
- способности описаны протоколами `Movable`, `Drawable`, `Collidable`;
- 15+ юнит-тестов защищают контракты.

## Как это подготовит следующие модули

- **Модуль 3 (SOLID):** ABC и протоколы — основа принципа инверсии зависимостей. Мы будем опираться на них при рефакторинге.
- **Модуль 4 (Game Loop, Component):** `GameObject` станет базой для компонентного подхода. `update(dt)` и `draw()` — ключевые методы игрового цикла.
- **Модуль 5 (State, Strategy, Command):** протоколы позволят подменять стратегии без наследования.
- **Модуль 6 (Observer, Factory):** dataclass-события и фабрики на основе конфигов.
- **Модуль 7 (ECS):** `GameObject` превратится в Entity, а компоненты — в dataclass-классы.

## Литература

**Базовая:**
- Лутц М. «Изучаем Python», главы 26–27.
- [Python docs: abc](https://docs.python.org/3/library/abc.html)
- [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
- [Python docs: typing.Protocol](https://typing.python.org/en/latest/spec/protocol.html)
- [Python docs: logging](https://docs.python.org/3/library/logging.html)

**Углублённая:**
- Ramalho L. «Fluent Python», 2-е изд. — главы о протоколах, дескрипторах и MRO.
- [Real Python: Implementing Interfaces in Python](https://realpython.com/python-interface/)
- [Real Python: Data Classes](https://realpython.com/python-data-classes/)
- [Real Python: Structural Subtyping with Protocols](https://realpython.com/python-protocol/)
- [Real Python: Multiple Inheritance and Mixins](https://realpython.com/inheritance-composition-python/)
- [PEP 544 — Protocols](https://peps.python.org/pep-0544/)
- [PEP 557 — Data Classes](https://peps.python.org/pep-0557/)
- [Python Cookbook: Extending Classes with Mixins](https://python-cookbook.readthedocs.io/zh-cn/stable/c08/p18_extending_classes_with_mixins.html)

## Что дальше

В модуле 3 мы изучим принципы SOLID и рефакторинг. Проект «Тайна исчезнувшего дива» продолжит развиваться: мы выделим ответственности классов, применим инверсию зависимостей и подготовим архитектуру к паттернам проектирования.